# 🧪 Colab 1 — LangChain Agent with Short-Term & Long-Term Memory + Tools

**Workshop: Agentic AI — Full-Day Training**

---

## Learning Objectives

By the end of this lab you will be able to:
1. Understand the difference between **Short-Term Memory (STM)** and **Long-Term Memory (LTM)** in LangChain agents
2. Implement `ConversationBufferMemory` (STM) and a `ChromaDB vector store` (LTM)
3. Wire up real tools: **web search**, **Python REPL**, and a **custom calculator**
4. Build and run a **ReAct agent** that reasons, acts, and remembers
5. *(Extension)* Swap STM for `ConversationSummaryMemory`, add SQL retrieval, self-critique loop, and streaming UI

---

## Architecture Overview

```
User Prompt
     │
     ▼
┌─────────────────────────────────────────────┐
│               AgentExecutor                  │
│                                             │
│  ┌─────────────┐    ┌────────────────────┐  │
│  │  ReAct LLM  │◄──►│  STM (Buffer/      │  │
│  │  (Claude /  │    │  Summary Memory)   │  │
│  │  GPT-4o)    │    └────────────────────┘  │
│  └──────┬──────┘                            │
│         │ tool calls                        │
│  ┌──────▼──────────────────────────────┐    │
│  │            Tool Router              │    │
│  │  [Search]  [PythonREPL]  [Calc]     │    │
│  └─────────────────────────────────────┘    │
└─────────────────────────────────────────────┘
     │
     ▼
┌─────────────────────┐
│  LTM: Chroma VectorDB│  ← stores every Q&A pair
│  (persistent across  │    retrieved at query time
│   sessions)         │
└─────────────────────┘
```

**STM** = the rolling conversation window this session  
**LTM** = semantic vector store that persists across sessions

---

## ⏱ Timing
| Section | Time |
|---------|------|
| Setup & Install | 10 min |
| Part 1 — STM Agent | 20 min |
| Part 2 — LTM + STM Agent | 20 min |
| Part 3 — Full Agent with Tools | 15 min |
| Extension Tasks | 30+ min |


---
## 📦 Section 0 — Installation & Setup

In [ ]:
# Install all required packages
# Run this cell first — it takes ~2 minutes
!pip install -q \
    langchain \
    langchain-anthropic \
    langchain-community \
    langchain-experimental \
    langchain-chroma \
    langchain-openai \
    chromadb \
    tavily-python \
    sentence-transformers \
    tiktoken \
    faiss-cpu

print("✅ Packages installed")


In [ ]:
import os

# ─── API Keys ────────────────────────────────────────────────────────────────
# Set your keys here OR use Colab Secrets (recommended)
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
# os.environ["TAVILY_API_KEY"]    = userdata.get("TAVILY_API_KEY")

os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # ← replace
os.environ["TAVILY_API_KEY"]    = "tvly-..."      # ← replace

# ─── Model ───────────────────────────────────────────────────────────────────
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    temperature=0,
    max_tokens=2048,
)

print(f"✅ LLM ready: {llm.model}")


---
## 🧠 Part 1 — Short-Term Memory (STM)

**Short-Term Memory** lives in the LLM's context window.  
It keeps the current conversation history so the agent can refer back to earlier messages.

### Two STM strategies we'll try:
| Class | How it works | Best for |
|-------|-------------|----------|
| `ConversationBufferMemory` | Keeps **all** messages verbatim | Short sessions, debugging |
| `ConversationSummaryMemory` | Summarises older exchanges to save tokens | Long sessions, production |

We'll start with `ConversationBufferMemory`.


In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

# ── Build STM ────────────────────────────────────────────────────────────────
stm = ConversationBufferMemory(
    memory_key="history",   # key injected into the prompt template
    return_messages=True,   # return as list[BaseMessage] not a string
)

# Quick test with a simple ConversationChain (no tools yet)
conv_chain = ConversationChain(llm=llm, memory=stm, verbose=True)

print("=== Turn 1 ===")
r1 = conv_chain.predict(input="Hi! My name is Alex and I'm an ML engineer.")
print(r1)

print("\n=== Turn 2 ===")
r2 = conv_chain.predict(input="What did I just tell you about myself?")
print(r2)


In [ ]:
# ── Inspect what's stored in STM ─────────────────────────────────────────────
print("Messages in STM buffer:")
for msg in stm.chat_memory.messages:
    role = msg.__class__.__name__
    print(f"  [{role}] {msg.content[:120]}")


### 🔍 Observation
Notice how the agent correctly recalls "Alex" and "ML engineer" — those facts lived in the STM buffer.

**Limitation**: if the conversation runs long, the context window fills up and older facts are silently dropped.  
This is exactly why we need **Long-Term Memory**.


---
## 📚 Part 2 — Long-Term Memory (LTM) with ChromaDB

**Long-Term Memory** stores information in a **vector database** that persists across sessions.  
At each turn the agent:
1. Embeds the user's query
2. Retrieves the top-k most relevant past Q&A pairs
3. Injects them into the prompt as additional context

### Why vector search?
Semantic similarity — not keyword matching. A query about "revenue last quarter" will retrieve  
"Q3 sales figures" even if those exact words weren't used.


In [ ]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.memory import VectorStoreRetrieverMemory

# ── Embeddings (local, no API key needed) ────────────────────────────────────
# Using sentence-transformers/all-MiniLM-L6-v2 (fast, 384-dim)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# ── Persistent Chroma vector store ───────────────────────────────────────────
# persist_directory keeps the DB between Colab sessions if you mount Drive
vectorstore = Chroma(
    collection_name="agent_ltm",
    embedding_function=embeddings,
    persist_directory="./chroma_ltm",   # remove for in-memory only
)

# ── Wrap as a retriever ───────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

ltm = VectorStoreRetrieverMemory(
    retriever=retriever,
    memory_key="ltm_context",
    return_docs=False,   # return as formatted string
)

print(f"✅ LTM ready — collection: '{vectorstore._collection.name}'")
print(f"   Docs currently stored: {vectorstore._collection.count()}")


In [ ]:
# ── Manually seed some long-term memories ────────────────────────────────────
# In production these would be saved automatically after each agent run.

seed_memories = [
    {"input": "What is the user's name?",         "output": "Alex"},
    {"input": "What does the user work on?",       "output": "Machine Learning engineering at a fintech startup"},
    {"input": "What stack does the user prefer?",  "output": "Python, PyTorch, LangChain, Postgres"},
    {"input": "What project is the user working on?",
     "output": "Building a RAG pipeline over internal financial documents"},
    {"input": "What LLM provider does the user prefer?",
     "output": "Anthropic Claude for reasoning tasks, OpenAI for embeddings"},
]

for mem in seed_memories:
    ltm.save_context({"input": mem["input"]}, {"output": mem["output"]})

print(f"✅ Seeded {len(seed_memories)} memories")
print(f"   Total docs in LTM: {vectorstore._collection.count()}")


In [ ]:
# ── Test LTM retrieval ────────────────────────────────────────────────────────
query = "What framework does Alex use for building AI systems?"
retrieved = ltm.load_memory_variables({"prompt": query})

print(f"Query:  {query}\n")
print("Retrieved LTM context:")
print(retrieved["ltm_context"])


### ✅ What just happened?
1. We embedded the query: *"What framework does Alex use..."*
2. Chroma found the 3 most semantically similar stored Q&A pairs
3. They were returned as context to inject into the next prompt

**Try changing the query** — notice the retrieval changes based on meaning, not exact keywords.


---
## 🔧 Part 3 — Defining Tools

We'll give the agent three tools:

| Tool | What it does | When the agent uses it |
|------|-------------|------------------------|
| `TavilySearch` | Real-time web search | Current events, facts it doesn't know |
| `PythonREPL` | Execute arbitrary Python code | Calculations, data manipulation, plotting |
| `Calculator` | Safe arithmetic evaluation | Simple math without running full Python |


In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_experimental.tools import PythonREPLTool
from langchain.tools import Tool
import math, ast, operator

# ── Tool 1: Web Search ────────────────────────────────────────────────────────
search_tool = TavilySearchResults(
    max_results=4,
    description=(
        "Search the web for real-time information. "
        "Use for current events, recent data, or anything beyond your training cutoff."
    ),
)

# ── Tool 2: Python REPL ───────────────────────────────────────────────────────
python_tool = PythonREPLTool()
python_tool.description = (
    "Execute Python code in a sandboxed REPL. "
    "Use for calculations, data analysis, string manipulation, or any computation. "
    "Input must be valid Python. Print your results."
)

# ── Tool 3: Safe Calculator ───────────────────────────────────────────────────
def safe_calc(expression: str) -> str:
    """Evaluate a simple arithmetic expression safely (no exec/eval tricks)."""
    allowed_ops = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg,
    }
    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.n
        elif isinstance(node, ast.BinOp):
            return allowed_ops[type(node.op)](_eval(node.left), _eval(node.right))
        elif isinstance(node, ast.UnaryOp):
            return allowed_ops[type(node.op)](_eval(node.operand))
        else:
            raise ValueError(f"Unsupported expression: {ast.dump(node)}")
    try:
        tree = ast.parse(expression, mode="eval")
        result = _eval(tree.body)
        return str(round(result, 6))
    except Exception as e:
        return f"Error: {e}"

calculator_tool = Tool(
    name="Calculator",
    func=safe_calc,
    description=(
        "Evaluate arithmetic expressions: +, -, *, /, **. "
        "Input: a plain math expression like '(1200 * 1.15) / 12'. "
        "Use this for simple arithmetic; use PythonREPL for complex logic."
    ),
)

tools = [search_tool, python_tool, calculator_tool]
print(f"✅ {len(tools)} tools ready: {[t.name for t in tools]}")


---
## 🤖 Part 4 — Assembling the Full Agent (STM + LTM + Tools)

We now combine everything:
- **STM** (`ConversationBufferMemory`) — rolling conversation window
- **LTM** (`VectorStoreRetrieverMemory`) — semantic recall from past sessions
- **Tools** — search, code execution, calculator

We use a **custom prompt** that injects both memory types, then a `ReAct` agent loop.


In [ ]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain.prompts import PromptTemplate

# ── Custom ReAct prompt that uses BOTH memory types ───────────────────────────
REACT_TEMPLATE = """You are a helpful, knowledgeable AI research assistant with access to tools.

### Long-Term Memory (from past sessions)
{ltm_context}

### Current Conversation (Short-Term Memory)
{history}

### Available Tools
{tools}

### Tool Names
{tool_names}

### Instructions
- Reason step-by-step using the format below
- Use tools when you need real-time data or computation
- Reference Long-Term Memory when relevant to personalise your response
- Be concise but thorough

### Format (STRICT — always follow this)
Question: the input question
Thought: what you need to do
Action: the tool name (must be one of [{tool_names}])
Action Input: the input to the tool
Observation: the tool result
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: your complete response

Begin!

Question: {input}
Thought: {agent_scratchpad}
"""

prompt = PromptTemplate(
    input_variables=["input", "history", "ltm_context", "tools", "tool_names", "agent_scratchpad"],
    template=REACT_TEMPLATE,
)

print("✅ Custom ReAct prompt created")
print(f"   Input variables: {prompt.input_variables}")


In [ ]:
# ── STM for this session ──────────────────────────────────────────────────────
session_stm = ConversationBufferMemory(
    memory_key="history",
    return_messages=False,   # string format for ReAct template
    input_key="input",
    output_key="output",
)

# ── Build the ReAct agent ─────────────────────────────────────────────────────
agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)

# ── AgentExecutor wires it all together ───────────────────────────────────────
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=session_stm,           # STM is managed automatically
    max_iterations=8,             # safety: cap the ReAct loop
    handle_parsing_errors=True,   # recover gracefully from format errors
    verbose=True,                 # show the full Thought→Action→Observation trace
    return_intermediate_steps=True,
)

print("✅ AgentExecutor ready")
print(f"   Max iterations: {executor.max_iterations}")


In [ ]:
# ── Helper: inject LTM context before each run ───────────────────────────────
def run_agent(user_input: str, show_steps: bool = False) -> str:
    """
    Run the agent with both STM (auto) and LTM (injected from Chroma).
    
    Args:
        user_input: the user's question
        show_steps: if True, print intermediate tool steps
    Returns:
        the agent's final answer
    """
    # 1. Retrieve relevant LTM context for this query
    ltm_vars = ltm.load_memory_variables({"prompt": user_input})
    ltm_context = ltm_vars.get("ltm_context", "No relevant past context found.")
    
    # 2. Run the agent
    result = executor.invoke({
        "input": user_input,
        "ltm_context": ltm_context,
    })
    
    # 3. Save this exchange to LTM for future sessions
    ltm.save_context(
        {"input": user_input},
        {"output": result["output"]},
    )
    
    # 4. Optional: show intermediate steps
    if show_steps and "intermediate_steps" in result:
        print("\n📋 Tool calls made:")
        for action, observation in result["intermediate_steps"]:
            print(f"  🔧 {action.tool}({action.tool_input!r})")
            print(f"     → {str(observation)[:200]}")
    
    return result["output"]

print("✅ run_agent() helper ready")


---
## 🧪 Part 5 — Test the Agent

Run the cells below one at a time. Observe:
- How the **LTM context** is injected at the top of each run
- The **Thought → Action → Observation** loop in verbose output
- How **STM** makes follow-up questions work naturally


In [ ]:
# ── Test 1: Personalised response using LTM ───────────────────────────────────
print("=" * 60)
print("TEST 1: Does the agent remember who it's talking to?")
print("=" * 60)

answer = run_agent(
    "What AI framework should I use for my project?",
    show_steps=True
)
print("\n🤖 Final Answer:")
print(answer)


In [ ]:
# ── Test 2: Multi-step with web search + calculation ─────────────────────────
print("=" * 60)
print("TEST 2: Web search + arithmetic in one task")
print("=" * 60)

answer = run_agent(
    "What is the current population of India? "
    "Calculate what 0.5% of that would be, and convert to millions.",
    show_steps=True
)
print("\n🤖 Final Answer:")
print(answer)


In [ ]:
# ── Test 3: STM in action — follow-up question ───────────────────────────────
print("=" * 60)
print("TEST 3: Follow-up using STM (no re-stating context)")
print("=" * 60)

run_agent("Tell me about the latest developments in transformer architectures.")
answer = run_agent("Which of those would be most relevant to my work?")  # refers to Test 1 LTM + Test 3 STM

print("\n🤖 Final Answer:")
print(answer)


In [ ]:
# ── Test 4: Python REPL for data analysis ────────────────────────────────────
print("=" * 60)
print("TEST 4: Agent writes and runs Python code")
print("=" * 60)

answer = run_agent(
    "Generate a list of the first 10 Fibonacci numbers, "
    "compute their sum, and tell me what percentage each number "
    "contributes to the total.",
    show_steps=True
)
print("\n🤖 Final Answer:")
print(answer)


In [ ]:
# ── Inspect STM after 4 turns ─────────────────────────────────────────────────
print("\n📝 Current STM buffer (last 4 turns):")
history_str = session_stm.load_memory_variables({})["history"]
print(history_str[:2000])

print(f"\n📚 LTM now contains {vectorstore._collection.count()} documents")


---
## 🔬 Part 6 — Observe & Compare: STM vs LTM

Run this cell to see the difference between what's in STM vs LTM right now.


In [ ]:
from IPython.display import Markdown, display

# ── What's in STM right now? ─────────────────────────────────────────────────
stm_history = session_stm.load_memory_variables({})["history"]

# ── What would LTM retrieve for a given query? ────────────────────────────────
test_query = "What programming languages does Alex use?"
ltm_result = ltm.load_memory_variables({"prompt": test_query})["ltm_context"]

summary_md = f"""
## Memory Comparison

### 🧠 Short-Term Memory (this session)
Contains **{len(session_stm.chat_memory.messages)} messages** from the current conversation.

```
{stm_history[-800:] if len(stm_history) > 800 else stm_history}
```

---

### 📚 Long-Term Memory (Chroma — persists across sessions)
Query: *"{test_query}"*

Retrieved context:
```
{ltm_result}
```

**Total documents in LTM:** {vectorstore._collection.count()}

---

### Key Differences
| Property | STM (Buffer) | LTM (Chroma) |
|----------|-------------|--------------|
| Scope | This session only | Across all sessions |
| Retrieval | Sequential (all messages) | Semantic similarity search |
| Persistence | Lost on session end | Saved to disk |
| Token cost | Grows linearly | Fixed-size injection (top-k) |
| Best for | Context continuity | User facts, past decisions |
"""

display(Markdown(summary_md))


---
---
# ⚡ Extension Tasks

These tasks are for participants who finish early. Each builds on the core agent above.

---

## ⚡ Extension 1 — Swap to `ConversationSummaryMemory`

`ConversationBufferMemory` keeps *all* messages verbatim.  
For long sessions this wastes tokens. `ConversationSummaryMemory` asks the LLM to  
**summarise older exchanges** and only keeps the summary + last N messages.


In [ ]:
# ── Extension 1: ConversationSummaryMemory ────────────────────────────────────
from langchain.memory import ConversationSummaryMemory

summary_stm = ConversationSummaryMemory(
    llm=llm,                    # LLM used to write the summary
    memory_key="history",
    return_messages=False,
    input_key="input",
    output_key="output",
    human_prefix="User",
    ai_prefix="Agent",
)

# Re-build the executor with summary memory
summary_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=summary_stm,
    max_iterations=8,
    handle_parsing_errors=True,
    verbose=False,              # quiet mode so we can focus on memory
    return_intermediate_steps=False,
)

def run_agent_summary(user_input: str) -> str:
    ltm_vars = ltm.load_memory_variables({"prompt": user_input})
    result = summary_executor.invoke({
        "input": user_input,
        "ltm_context": ltm_vars.get("ltm_context", ""),
    })
    return result["output"]

# ── Run several turns and watch the summary grow ──────────────────────────────
for turn in [
    "Tell me about the history of neural networks.",
    "What were the key innovations in the 2010s?",
    "How did attention mechanisms change everything?",
    "What should I read to go deeper on this?",
]:
    run_agent_summary(turn)

# ── Inspect the running summary ───────────────────────────────────────────────
print("📝 Running conversation summary (STM):")
print(summary_stm.moving_summary_buffer or "(no summary yet — buffer not full)")
print(f"\nMessages still in buffer: {len(summary_stm.chat_memory.messages)}")


### 💡 Discussion
- How does the summary compare to the raw buffer from Part 5?  
- What information was compressed or lost?  
- When would you choose Summary over Buffer in production?


---
## ⚡ Extension 2 — Add a SQLite Tool (Structured LTM)

Vector search is great for semantic retrieval, but sometimes you need **exact lookups** —  
user IDs, transaction amounts, timestamps. A SQL tool gives the agent structured memory.


In [ ]:
# ── Extension 2: SQLite as structured LTM ────────────────────────────────────
import sqlite3, json
from langchain.tools import StructuredTool
from pydantic import BaseModel

# ── Create a simple SQLite DB ─────────────────────────────────────────────────
conn = sqlite3.connect(":memory:")   # use a file path for persistence
cur = conn.cursor()

cur.executescript("""
    CREATE TABLE IF NOT EXISTS user_preferences (
        id INTEGER PRIMARY KEY,
        category TEXT,
        key TEXT,
        value TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    INSERT INTO user_preferences (category, key, value) VALUES
        ('tools',    'preferred_llm',       'Claude Sonnet'),
        ('tools',    'preferred_framework', 'LangChain + LangGraph'),
        ('project',  'name',                'FinDoc RAG Pipeline'),
        ('project',  'tech_stack',          'Python, Chroma, FastAPI'),
        ('project',  'deadline',            '2025-09-30');
""")
conn.commit()

# ── Tool: SQL query ───────────────────────────────────────────────────────────
class SQLQueryInput(BaseModel):
    query: str

def run_sql(query: str) -> str:
    """Run a read-only SQL query against the user preferences DB."""
    try:
        # Safety: only allow SELECT
        if not query.strip().upper().startswith("SELECT"):
            return "Error: only SELECT queries are allowed."
        rows = cur.execute(query).fetchall()
        cols = [d[0] for d in cur.description]
        if not rows:
            return "No results found."
        return json.dumps([dict(zip(cols, row)) for row in rows], indent=2)
    except Exception as e:
        return f"SQL Error: {e}"

sql_tool = StructuredTool.from_function(
    func=run_sql,
    name="UserPreferencesDB",
    description=(
        "Query the user's structured preference database using SQL SELECT statements. "
        "Table: user_preferences(id, category, key, value, created_at). "
        "Use this to look up exact user settings, project details, or tool preferences."
    ),
    args_schema=SQLQueryInput,
)

# ── Rebuild with 4 tools ──────────────────────────────────────────────────────
extended_tools = [search_tool, python_tool, calculator_tool, sql_tool]

ext_agent = create_react_agent(llm=llm, tools=extended_tools, prompt=prompt)
ext_executor = AgentExecutor(
    agent=ext_agent,
    tools=extended_tools,
    memory=ConversationBufferMemory(memory_key="history", return_messages=False,
                                    input_key="input", output_key="output"),
    max_iterations=8,
    handle_parsing_errors=True,
    verbose=True,
)

# ── Test it ───────────────────────────────────────────────────────────────────
def run_ext(q):
    ltm_ctx = ltm.load_memory_variables({"prompt": q}).get("ltm_context", "")
    return ext_executor.invoke({"input": q, "ltm_context": ltm_ctx})["output"]

print(run_ext("What framework am I using and when is my project deadline?"))


---
## ⚡ Extension 3 — Self-Critique Loop (Reflexion Pattern)

The **Reflexion** pattern asks the agent to evaluate its own answer,  
identify weaknesses, then produce an improved version.


In [ ]:
# ── Extension 3: Self-Critique Loop ──────────────────────────────────────────
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── Critic prompt ─────────────────────────────────────────────────────────────
critic_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a rigorous AI quality reviewer.
    
Given a question and an agent's answer, provide:
1. A score from 1-10 (10 = perfect)
2. Specific weaknesses (missing facts, logic errors, unclear language)
3. A concrete suggestion for improvement

Format:
SCORE: <number>
WEAKNESSES: <bullet points>
SUGGESTION: <one clear improvement instruction>
"""),
    ("human", "Question: {question}\n\nAnswer: {answer}"),
])

# ── Revision prompt ───────────────────────────────────────────────────────────
revision_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a precise AI assistant. Revise the answer based on the feedback provided."),
    ("human", "Original question: {question}\n\nOriginal answer: {answer}\n\nCritic feedback: {critique}\n\nRevised answer:"),
])

critic_chain = critic_prompt | llm | StrOutputParser()
revision_chain = revision_prompt | llm | StrOutputParser()

def reflexion_run(question: str, max_rounds: int = 2) -> dict:
    """Run the agent, then apply self-critique rounds."""
    print(f"\n{'='*55}")
    print(f"Question: {question}")
    print('='*55)
    
    # Initial answer from the agent
    ltm_ctx = ltm.load_memory_variables({"prompt": question}).get("ltm_context", "")
    initial = executor.invoke({"input": question, "ltm_context": ltm_ctx})["output"]
    print(f"\n[Round 0 — Initial Answer]\n{initial}")
    
    current_answer = initial
    history = [{"round": 0, "answer": initial, "score": None, "critique": None}]
    
    for rnd in range(1, max_rounds + 1):
        # Critique
        critique = critic_chain.invoke({"question": question, "answer": current_answer})
        score_line = [l for l in critique.split("\n") if l.startswith("SCORE:")]
        score = int(score_line[0].split(":")[1].strip()) if score_line else 0
        print(f"\n[Round {rnd} — Critique] Score: {score}/10")
        print(critique)
        
        if score >= 9:
            print("\n✅ Score threshold reached — stopping early.")
            break
        
        # Revise
        current_answer = revision_chain.invoke({
            "question": question,
            "answer": current_answer,
            "critique": critique,
        })
        print(f"\n[Round {rnd} — Revised Answer]\n{current_answer}")
        history.append({"round": rnd, "answer": current_answer, "score": score, "critique": critique})
    
    return {"final_answer": current_answer, "history": history}

# ── Run it ────────────────────────────────────────────────────────────────────
result = reflexion_run(
    "What are the main risks of using LLM-based agents in production financial systems?",
    max_rounds=2
)


### 💡 Discussion
- Did the score improve across rounds?
- What kinds of weaknesses did the critic identify?
- Is there a point of diminishing returns? When would you cap the rounds?


---
## ⚡ Extension 4 — Streaming Agent Output to a UI

In production you want to **stream** the agent's intermediate steps to the user  
so they see progress rather than a blank screen for 30 seconds.


In [ ]:
# ── Extension 4: Streaming with callbacks ─────────────────────────────────────
from langchain.callbacks.base import BaseCallbackHandler
from langchain.callbacks import StdOutCallbackHandler
import time

class StreamingDisplayHandler(BaseCallbackHandler):
    """Custom callback that prints each token/step as it arrives."""

    def on_llm_new_token(self, token: str, **kwargs):
        print(token, end="", flush=True)

    def on_tool_start(self, serialized, input_str, **kwargs):
        tool_name = serialized.get("name", "unknown")
        print(f"\n\n🔧 Calling tool: {tool_name}")
        print(f"   Input: {str(input_str)[:150]}")

    def on_tool_end(self, output, **kwargs):
        print(f"   Result: {str(output)[:200]}")

    def on_agent_action(self, action, **kwargs):
        print(f"\n💭 Thought → {action.log[:300]}")

    def on_agent_finish(self, finish, **kwargs):
        print(f"\n\n✅ Final Answer: {finish.output}")

# ── Build a streaming executor ────────────────────────────────────────────────
streaming_llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    temperature=0,
    streaming=True,
    callbacks=[StreamingDisplayHandler()],
)

stream_agent = create_react_agent(llm=streaming_llm, tools=tools, prompt=prompt)
stream_executor = AgentExecutor(
    agent=stream_agent,
    tools=tools,
    memory=ConversationBufferMemory(memory_key="history", return_messages=False,
                                    input_key="input", output_key="output"),
    max_iterations=6,
    handle_parsing_errors=True,
    verbose=False,   # using our custom handler instead
)

print("🚀 Running agent with live streaming output...\n")
ltm_ctx = ltm.load_memory_variables({"prompt": "streaming test"}).get("ltm_context", "")
stream_executor.invoke({
    "input": "Search for the latest news about LangChain updates and summarise the top 3 items.",
    "ltm_context": ltm_ctx,
})


---
## 🎯 Lab Summary

### What you built

| Component | Class / Tool | Purpose |
|-----------|-------------|---------|
| **STM (Buffer)** | `ConversationBufferMemory` | Full conversation history this session |
| **STM (Summary)** | `ConversationSummaryMemory` | Compressed history — saves tokens |
| **LTM** | `VectorStoreRetrieverMemory` + Chroma | Semantic recall across sessions |
| **Structured LTM** | SQLite + `StructuredTool` | Exact lookup of user facts |
| **Web Search** | `TavilySearchResults` | Real-time grounded answers |
| **Code Execution** | `PythonREPLTool` | Dynamic computation |
| **Calculator** | Custom `Tool` | Safe arithmetic |
| **Self-Critique** | Reflexion chain | Iterative quality improvement |
| **Streaming** | `BaseCallbackHandler` | Live progress display |

---

### Key Takeaways

1. **STM ≠ LTM** — they serve different purposes and should be used together
2. **Buffer vs Summary** — choose based on session length and token budget  
3. **LTM requires a retrieval strategy** — semantic (Chroma) or structured (SQL)  
4. **Reflexion improves quality** — at the cost of latency and tokens  
5. **Streaming is a UX necessity** — always add it before going to production

---

### Next Steps
- 📘 Colab 2: Rebuild this agent natively with the Anthropic SDK (no LangChain)
- 🔁 Compare: latency, cost, reasoning trace quality between the two approaches
- 🚀 Advanced: deploy with LangGraph + `MemorySaver` for production-grade persistence
